In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

Reading Bronze data

In [0]:
customers_bronze_df = spark.table(
    "workspace.bronze.customers"
)

customers_bronze_df.show(10)

Standardize Text columns

In [0]:
customers_cleaned_df = (
    customers_bronze_df.withColumn("name", trim(col("name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", trim(col("city")))
)

Creating Validation Flags for columnns

In [0]:
customers_validated_df = (
    customers_cleaned_df
    .withColumn("invalid_customer_id", when(col("customer_id").isNull(), True).otherwise(False))
    .withColumn("invalid_name", when(col("name").isNull() | (col("name") == ""),True).otherwise(False))
    .withColumn("invalid_age", when(col("age").isNull() | (col("age") < 18) | (col("age") > 100), True).otherwise(False))
    .withColumn("invalid_email", when(col("email").isNull() | ~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), True).otherwise(False))
)

In [0]:
customers_validated_df = (
    customers_validated_df
    .withColumn(
        "is_valid",
        ~(
            col("invalid_customer_id") |
            col("invalid_name") |
            col("invalid_age") |
            col("invalid_email")
        )
    )
)

In [0]:
customers_validated_df.select(
    "customer_id",
    "name",
    "email",
    "age",
    "invalid_age",
    "invalid_email",
    "is_valid"
).show(20, truncate=False)

In [0]:
customers_validated_df.groupBy(
    "is_valid"
).count().show()

Data Completeness Score

In [0]:
customers_scored_df = (
    customers_validated_df
    .withColumn(
        "completeness_score",
        (
            when(col("customer_id").isNotNull(), 1).otherwise(0)
            +
            when(col("name").isNotNull(), 1).otherwise(0)
            +
            when(col("email").isNotNull(), 1).otherwise(0)
            +
            when(col("city").isNotNull(), 1).otherwise(0)
            +
            when(col("age").isNotNull(), 1).otherwise(0)
        )
    )
)

In [0]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        col("completeness_score").desc(),
        col("ingestion_timestamp").desc()
    )
)

In [0]:
customers_ranked_df = (
    customers_scored_df
    .withColumn(
        "row_number",
        row_number().over(customer_window)
    )
)

Keeping the record where row_number=1

In [0]:
customers_deduped_df = (
    customers_ranked_df
    .filter(
        col("row_number") == 1
    )
)

Quality Check

In [0]:
customers_deduped_df.groupBy(
    "customer_id"
).count().filter(
    col("count") > 1
).show()

Remoing Unecessary columns

In [0]:
silver_customers_df = (
    customers_deduped_df
    .select(
        "customer_id",
        "name",
        "email",
        "city",
        "age",
        "ingestion_timestamp",
        "load_date",
        "source_file"
    )
)

Storing Invalid Records

In [0]:
customer_rejects_df = (
    customers_validated_df
    .filter(
        ~col("is_valid")
    )
)

In [0]:
customer_rejects_df = (
    customer_rejects_df
    .withColumn(
        "reject_reason",
        when(
            col("invalid_customer_id"),
            "Invalid customer_id"
        )
        .when(
            col("invalid_name"),
            "Missing customer name"
        )
        .when(
            col("invalid_age"),
            "Invalid age"
        )
        .when(
            col("invalid_email"),
            "Invalid email"
        )
        .otherwise(
            "Unknown validation error"
        )
    )
)

In [0]:
customer_rejects_df.select(
    "customer_id",
    "name",
    "email",
    "age",
    "reject_reason"
).show(
    20,
    truncate=False
)

Silver Customers

In [0]:
(
    silver_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.customers"
    )
)

Silver Reject Customers

In [0]:
(
    customer_rejects_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.customer_rejects"
    )
)

In [0]:
print(
    "Silver customers:",
    spark.table(
        "workspace.silver.customers"
    ).count()
)

print(
    "Customer rejects:",
    spark.table(
        "workspace.silver.customer_rejects"
    ).count()
)